# Notebook 19 — Recovery-regime audit (G4)

Audits all 15 calibrated students (5 methods x 3 realized budgets) under three recovery regimes: none, minimal (1 epoch, seeded 10% train subset, class-weighted CE), and full (merged from the NB17 calibrated screen). Tests the pivoted claim: semantic selection matters when recovery is constrained. Resumable per (method, budget, regime).

In [ ]:
# NB19: recovery-regime audit of all 15 calibrated students.
# Regimes: none (raw pruned), minimal (1 epoch on a seeded 10% train subset),
# full (already computed in NB17-calibrated; merged, not retrained).
# Pre-registered gate G4: saber_v2 has strictly the most best-method cells on
# BOTH awbir (lower better) and family_macro_f1 (higher better) across the
# 6 constrained-recovery cells (3 budgets x {none, minimal}).
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import yaml

REPO = Path("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
os.chdir(REPO); sys.path.insert(0, str(REPO))

from src.saber.bridge_ciciot import load_bridge
from src.saber.taxonomy import ciciot2023_taxonomy, DEFAULT_COST_PROFILES
from src.saber.surgery import prune_cnn1d_channels, count_parameters
from src.saber.metrics import full_model_audit, action_weighted_boundary_inversion_rate

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SABER_CFG = yaml.safe_load(open(REPO / "config/saber.yaml"))
MIN_W = int(SABER_CFG["groups"]["minimum_remaining_per_layer"])

OUTPUT_ROOT = REPO / "results/saber"
SELECT_DIR = OUTPUT_ROOT / "17_structured_selection"
OUT = OUTPUT_ROOT / "19_recovery_regimes"
OUT.mkdir(parents=True, exist_ok=True)

TRAIN_LOADER, VAL_LOADER, TEST_LOADER, MODEL, CLASS_NAMES = load_bridge()
MODEL = MODEL.to(DEVICE).eval()
taxonomy = ciciot2023_taxonomy(CLASS_NAMES)
robust_graph = pd.read_csv(OUTPUT_ROOT / "14_risk_graph" / "asvg_edges_robust.csv")

cached = np.load(OUTPUT_ROOT / "14_risk_graph" / "validation_teacher_outputs.npz",
                 allow_pickle=True)
TEACHER_LOGITS = cached["logits"]; VAL_LABELS = cached["labels"].astype(np.int64)

xb, _ = next(iter(VAL_LOADER))
EXAMPLE_INPUT = xb[:8].float().to(DEVICE)

METHODS = ["random", "magnitude", "taylor", "fisher", "saber_v2"]
BUDGETS = [0.25, 0.40, 0.55]
HKEYS = ["fine_macro_f1", "family_macro_f1", "attack_to_benign_rate",
         "benign_to_attack_rate", "hsr_balanced_soc", "ece15"]

def evaluate(model):
    model = model.to(DEVICE).eval()
    outs = []
    with torch.no_grad():
        for x, _ in VAL_LOADER:
            outs.append(model(x.float().to(DEVICE)).cpu().numpy())
    logits = np.concatenate(outs)
    audit = full_model_audit(logits, VAL_LABELS, taxonomy, DEFAULT_COST_PROFILES)
    awbir, _ = action_weighted_boundary_inversion_rate(
        TEACHER_LOGITS, logits, VAL_LABELS, robust_graph)
    audit["awbir"] = float(awbir)
    return audit

def rebuild(method, budget):
    tag = f"{method}_r{int(round(100*budget)):02d}cal"
    removed = pd.read_csv(SELECT_DIR / f"{tag}_removed_groups.csv")
    pm = {str(l): sorted(g["channel_index"].astype(int).tolist())
          for l, g in removed.groupby("module_path")}
    student, _ = prune_cnn1d_channels(
        MODEL, pm, EXAMPLE_INPUT, minimum_remaining_per_layer=MIN_W)
    return student.to(DEVICE)

# Shared minimal-recovery ingredients (identical for every student)
train_y = TRAIN_LOADER.dataset.tensors[1].numpy()
counts = np.bincount(train_y, minlength=taxonomy.n_classes)
w = np.zeros_like(counts, dtype=np.float64)
w[counts > 0] = 1.0 / np.sqrt(counts[counts > 0]); w[counts > 0] /= w[counts > 0].mean()
CLASS_W = torch.tensor(w, dtype=torch.float32, device=DEVICE)

g = torch.Generator().manual_seed(0)
n_train = len(TRAIN_LOADER.dataset)
SUB_IDX = torch.randperm(n_train, generator=g)[: n_train // 10]
SUB_LOADER = torch.utils.data.DataLoader(
    torch.utils.data.Subset(TRAIN_LOADER.dataset, SUB_IDX.tolist()),
    batch_size=1024, shuffle=True,
    generator=torch.Generator().manual_seed(0))

def minimal_finetune(student):
    student = student.to(DEVICE).train()
    opt = torch.optim.Adam(student.parameters(), lr=1e-3)
    lossf = nn.CrossEntropyLoss(weight=CLASS_W)
    for x, y in SUB_LOADER:
        opt.zero_grad()
        loss = lossf(student(x.float().to(DEVICE)), y.to(DEVICE))
        loss.backward(); opt.step()
    return student.eval()

RESULT_CSV = OUT / "recovery_regime_audit.csv"
if RESULT_CSV.exists():
    rows = pd.read_csv(RESULT_CSV).to_dict(orient="records")
    done = {(r["method"], r["budget"], r["regime"]) for r in rows}
else:
    rows, done = [], set()

for method in METHODS:
    for budget in BUDGETS:
        for regime in ["none", "minimal"]:
            if (method, budget, regime) in done:
                continue
            torch.manual_seed(0)
            student = rebuild(method, budget)
            if regime == "minimal":
                student = minimal_finetune(student)
            audit = evaluate(student)
            row = {"method": method, "budget": budget, "regime": regime,
                   "parameters": count_parameters(student),
                   "awbir": audit["awbir"],
                   **{k: float(audit[k]) for k in HKEYS}}
            rows.append(row)
            pd.DataFrame(rows).to_csv(RESULT_CSV, index=False)
            print(f"{method} b{budget} {regime}: awbir={row['awbir']:.4f} "
                  f"famF1={row['family_macro_f1']:.4f} "
                  f"a2b={row['attack_to_benign_rate']:.4f} "
                  f"b2a={row['benign_to_attack_rate']:.4f}")

df = pd.DataFrame(rows)

# Merge the full-recovery arm from the calibrated screen (no retraining)
cal = pd.read_csv(SELECT_DIR / "structured_screening_results_calibrated.csv")
full = cal.rename(columns={
    "post_awbir": "awbir", "post_macro_f1": "fine_macro_f1",
    "post_family_macro_f1": "family_macro_f1",
    "post_attack_to_benign": "attack_to_benign_rate",
    "post_benign_to_attack": "benign_to_attack_rate",
    "post_hsr_balanced_soc": "hsr_balanced_soc", "post_ece15": "ece15"})
full["regime"] = "full"
full = full[["method", "budget", "regime", "awbir"] + HKEYS]
combined = pd.concat([df[["method", "budget", "regime", "awbir"] + HKEYS], full])
combined.to_csv(OUT / "recovery_regime_combined.csv", index=False)

# Gate G4 over the 6 constrained cells
cells = [(b, r) for b in BUDGETS for r in ["none", "minimal"]]
wins = {m: {"awbir": 0, "family_macro_f1": 0} for m in METHODS}
for b, r in cells:
    sub = df[(df["budget"] == b) & (df["regime"] == r)]
    wins[sub.loc[sub["awbir"].idxmin(), "method"]]["awbir"] += 1
    wins[sub.loc[sub["family_macro_f1"].idxmax(), "method"]]["family_macro_f1"] += 1
sab = wins["saber_v2"]
others_awbir = max(v["awbir"] for m, v in wins.items() if m != "saber_v2")
others_fam = max(v["family_macro_f1"] for m, v in wins.items() if m != "saber_v2")
passed = bool(sab["awbir"] > others_awbir and sab["family_macro_f1"] > others_fam)

def to_py(o):
    if isinstance(o, dict): return {k: to_py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [to_py(v) for v in o]
    if isinstance(o, np.generic): return o.item()
    return o

gate = {"gate": "G4_recovery_constrained_regime", "passed": passed,
        "wins_per_method": wins,
        "criterion": "saber_v2 strictly most best-cells on awbir AND family_macro_f1 "
                     "over 3 budgets x {none, minimal}"}
(OUT / "G4_regime_gate.json").write_text(json.dumps(to_py(gate), indent=2))
print(json.dumps(to_py(gate), indent=2))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for j, (metric, better) in enumerate([("awbir", "lower"), ("family_macro_f1", "higher")]):
    for i, regime in enumerate(["none", "minimal"]):
        ax = axes[i][j]
        for m in METHODS:
            sub = df[(df["method"] == m) & (df["regime"] == regime)].sort_values("budget")
            ax.plot(sub["budget"], sub[metric], marker="o", label=m)
        ax.set_title(f"{metric} ({better} better) - {regime} recovery")
        ax.set_xlabel("realized FLOP reduction"); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "regime_pareto.png", dpi=120); plt.show()
print("Saved to", OUT)
